In [1]:
import pandas as pd

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)


In [12]:
import pandas as pd
import os

occupancy_dir = "/workspaces/CUBES/cubes/data/occupants"
files = [f for f in os.listdir(occupancy_dir) if f.startswith("rep_") and f.endswith(".sch")]

for file in files:
    print(file)
    path = os.path.join(occupancy_dir, file)
    df = pd.read_csv(path, index_col=0)

    df_clipped = df.clip(0, 1)  # Clip values between 0 and 1

    # Calculate row sums
    row_sums = df_clipped.sum(axis=1)

    # Avoid division by zero by replacing zeros in row_sums with 1
    row_sums = row_sums.replace(0, 1)

    # Fractionalise each row using broadcasting
    df_fractionalised = df_clipped.div(row_sums, axis=0)

    # Save back to the same file
    df_fractionalised.to_csv(path, index=False)


rep_6.sch
rep_7.sch
rep_5.sch
rep_4.sch
rep_1.sch
rep_3.sch
rep_2.sch
rep_10.sch
rep_9.sch
rep_8.sch


In [13]:
f = pd.read_csv("/workspaces/CUBES/cubes/data/occupants/rep_1.sch")

In [14]:
f

,front_room,kitchen,backroom,hall_downstairs,bedroom_1,bedroom_2,bedroom_3,bathroom,hall_upstairs
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
525595,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
525596,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
525597,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525598,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
df

,0,1,2,3,4,5,6,7,8,9
0,UTC_Time,front_room,kitchen,backroom,hall_downstairs,bedroom_1,bedroom_2,bedroom_3,bathroom,hall_upstairs
1,2013-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2013-01-01 00:01:00,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,2013-01-01 00:02:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2013-01-01 00:03:00,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
525596,2013-12-31 23:55:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525597,2013-12-31 23:56:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
525598,2013-12-31 23:57:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
525599,2013-12-31 23:58:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
occ = pd.read_csv("/workspaces/CUBES/cubes/data/occupants/thermostat_experiment/rep_0.sch")

In [8]:

# Assuming 'occ' is your original DataFrame
df = pd.DataFrame()
df["UTC_Time"] = pd.to_datetime(occ["UTC_Time"])  # Ensure UTC_Time is in datetime format

# Define the schedules
once_code_schedule = [(6, 23)]                      # 17 hours
twice_code_schedule = [(6, 9), (15, 23)]            # 11 hours
thrice_code_schedule = [(6, 8), (12, 14), (18, 23)] # 9 hours

# Create a function to adjust the schedule based on the day of the week
def adjust_schedule(day_name, onoff_times):
    if day_name > 4:
        if onoff_times == [(6, 9), (15, 23)]:              # 11 hours
            return [(7, 11), (15, 23)]                     # 12 hours
        elif onoff_times == [(6, 8), (12, 14), (18, 23)]:  # 9 hours
            return [(7, 10), (12, 14), (18, 23)]           # 10 hours
    return onoff_times

# Function to create the indicator based on the active schedule
def create_indicator_column(df, schedule):
    df['Hour'] = df['UTC_Time'].dt.hour
    df['Day'] = df['UTC_Time'].dt.weekday  # Monday=0, Sunday=6

    # Adjust the schedule for weekends and apply
    adjusted_schedule = [adjust_schedule(day, schedule) for day in df['Day']]

    # Create an indicator column
    indicator = [
        any(start <= hour < end for start, end in active_schedule)
        for hour, active_schedule in zip(df['Hour'], adjusted_schedule)
    ]
    return pd.Series(indicator).astype(int)

# Create columns for each code schedule
df['Once_Code'] = create_indicator_column(df, once_code_schedule)
df['Twice_Code'] = create_indicator_column(df, twice_code_schedule)
df['Thrice_Code'] = create_indicator_column(df, thrice_code_schedule)

# Drop unnecessary columns
df = df.drop(['Hour', 'Day'], axis=1)


In [9]:
df

,UTC_Time,Once_Code,Twice_Code,Thrice_Code
0,2013-01-01 00:00:00,0,0,0
1,2013-01-01 00:01:00,0,0,0
2,2013-01-01 00:02:00,0,0,0
3,2013-01-01 00:03:00,0,0,0
4,2013-01-01 00:04:00,0,0,0
...,...,...,...,...
525595,2013-12-31 23:55:00,0,0,0
525596,2013-12-31 23:56:00,0,0,0
525597,2013-12-31 23:57:00,0,0,0
525598,2013-12-31 23:58:00,0,0,0


In [11]:
once = pd.DataFrame(df[["UTC_Time", "Once_Code"]])
twice = pd.DataFrame(df[["UTC_Time", "Twice_Code"]])
thrice = pd.DataFrame(df[["UTC_Time", "Thrice_Code"]])

In [17]:
rooms = [
        "hall_downstairs",
        "front_room",
        "kitchen",
        "backroom",
        "bedroom_3",
        "bedroom_1",
        "hall_upstairs",
        "bathroom",
        "bedroom_2"
    ]
for room in rooms:
    once[room] = once["Once_Code"]

once = once.drop("Once_Code", axis=1)

In [114]:
once

,UTC_Time,hall_downstairs,front_room,kitchen,backroom,bedroom_3,bedroom_1,hall_upstairs,bathroom,bedroom_2
0,2013-01-01 00:00:00,0,0,0,0,0,0,0,0,0
1,2013-01-01 00:01:00,0,0,0,0,0,0,0,0,0
2,2013-01-01 00:02:00,0,0,0,0,0,0,0,0,0
3,2013-01-01 00:03:00,0,0,0,0,0,0,0,0,0
4,2013-01-01 00:04:00,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
525595,2013-12-31 23:55:00,0,0,0,0,0,0,0,0,0
525596,2013-12-31 23:56:00,0,0,0,0,0,0,0,0,0
525597,2013-12-31 23:57:00,0,0,0,0,0,0,0,0,0
525598,2013-12-31 23:58:00,0,0,0,0,0,0,0,0,0


In [15]:
twice

,UTC_Time,hall_downstairs,front_room,kitchen,backroom,bedroom_3,bedroom_1,hall_upstairs,bathroom,bedroom_2
0,2013-01-01 00:00:00,0,0,0,0,0,0,0,0,0
1,2013-01-01 00:01:00,0,0,0,0,0,0,0,0,0
2,2013-01-01 00:02:00,0,0,0,0,0,0,0,0,0
3,2013-01-01 00:03:00,0,0,0,0,0,0,0,0,0
4,2013-01-01 00:04:00,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
525595,2013-12-31 23:55:00,0,0,0,0,0,0,0,0,0
525596,2013-12-31 23:56:00,0,0,0,0,0,0,0,0,0
525597,2013-12-31 23:57:00,0,0,0,0,0,0,0,0,0
525598,2013-12-31 23:58:00,0,0,0,0,0,0,0,0,0


In [16]:
thrice

,UTC_Time,hall_downstairs,front_room,kitchen,backroom,bedroom_3,bedroom_1,hall_upstairs,bathroom,bedroom_2
0,2013-01-01 00:00:00,0,0,0,0,0,0,0,0,0
1,2013-01-01 00:01:00,0,0,0,0,0,0,0,0,0
2,2013-01-01 00:02:00,0,0,0,0,0,0,0,0,0
3,2013-01-01 00:03:00,0,0,0,0,0,0,0,0,0
4,2013-01-01 00:04:00,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
525595,2013-12-31 23:55:00,0,0,0,0,0,0,0,0,0
525596,2013-12-31 23:56:00,0,0,0,0,0,0,0,0,0
525597,2013-12-31 23:57:00,0,0,0,0,0,0,0,0,0
525598,2013-12-31 23:58:00,0,0,0,0,0,0,0,0,0


In [18]:
once.to_csv("/workspaces/CUBES/cubes/data/heating_pattern/manual_code/heating_once.sch",index=False)
twice.to_csv("/workspaces/CUBES/cubes/data/heating_pattern/manual_code/heating_twice.sch",index=False)
thrice.to_csv("/workspaces/CUBES/cubes/data/heating_pattern/manual_code/heating_thrice.sch",index=False)

In [1]:
import json
import os

def load_json(file_path):
    """Helper function to load a JSON file."""
    with open(file_path, 'r') as file:
        return json.load(file)

def deep_update(template, updates):
    """
    Recursively update the template with values from updates.
    - If a key in updates is a dictionary, it will update the sub-keys.
    - If a key in updates is a list, it will replace the list in the template.
    - If a key in updates is a scalar value, it will overwrite the value in the template.
    """
    for key, value in updates.items():
        if isinstance(value, dict) and key in template and isinstance(template[key], dict):
            deep_update(template[key], value)
        else:
            template[key] = value
    return template

def generate_building_config(year, case):

    # Load the main template configuration
    with open("constants/template_buildingconfig.json", "r") as file:
        config = json.load(file)

    # Load and update the schedules
    schedule_file = f"schedule_paths/{year}_files.json"
    if os.path.exists(schedule_file):
        schedule_details = load_json(schedule_file)
        config = deep_update(config, schedule_details)

    # Determine the construction and ventilation file paths based on the case
    if case in [0, 5, 10]:
        construction_path = 'constructions/1919.json'
        ventilation_path = "ventilation/natural.json"
    elif case in [1, 6, 11]:
        construction_path = 'constructions/1950.json'
        ventilation_path = "ventilation/natural.json"
    elif case in [2, 7, 12]:
        construction_path = 'constructions/1990.json'
        ventilation_path = "ventilation/natural.json"
    elif case in [3, 8, 13]:
        construction_path = 'constructions/2010.json'
        ventilation_path = "ventilation/natural.json"
    elif case in [4, 9, 14]:
        construction_path = 'constructions/Passivhaus.json'
        ventilation_path = "ventilation/natural.json"
    else:
        raise ValueError(f"Invalid case number: {case}. Must be between 0 and 14.")

    # Update construction details by matching entries
    if os.path.exists(construction_path):
        construction_details = load_json(construction_path)
        config = deep_update(config, construction_details)

    # Update ventilation details by matching entries
    if os.path.exists(ventilation_path):
        ventilation_details = load_json(ventilation_path)
        config = deep_update(config, ventilation_details)

    # Determine the energy system file path based on the case
    if 0 <= case <= 4:
        energy_system_path = "energy_system/boiler.json"
    elif 5 <= case <= 9:
        energy_system_path = "energy_system/heatpump.json"
    elif 10 <= case <= 14:
        energy_system_path = "energy_system/heatpump.json"
        pv_path = "energy_system/PV_Battery.json"
        if os.path.exists(pv_path):
            pv_details = load_json(pv_path)
            config = deep_update(config, pv_details)

    # Update energy system details by matching entries
    if os.path.exists(energy_system_path):
        energy_system_details = load_json(energy_system_path)
        config = deep_update(config, energy_system_details)

    # Update the occupant schedule, heating setpoint, and setback values in the template


    # Construct the output directory and filename based on `year`, `case`, and `zone`
    output_directory = f"/workspaces/CUBES/exp/jack/paper/thermostat_experiment/input/paper/{year}"
    os.makedirs(output_directory, exist_ok=True)  # Create the directories if they don't exist

    output_file = f"{output_directory}/case{case}.json"

    # Save the updated configuration to the specified path
    with open(output_file, "w") as file:
        json.dump(config, file, indent=4)

    return


In [2]:
import numpy as np

cases = np.arange(0,15)

for case in cases:
    generate_building_config(year=2023, case=case)